In [0]:
-- Step 1, 2, 3은 이전과 동일
-- Step 1: 월별 활동 기록 추출
WITH user_dately_activity AS (
    SELECT
        DISTINCT
        date_format(log_create_time, 'yyyy-MM-dd') AS activity_date,
        device_name,
        mac_addr
    FROM kic_data_ods.tlamp.normal_log_webos24
    WHERE 1=1
        AND date_ym = '2025-05'
        AND DEVICE_NAME in ('k24', 'k8lpn2', 'o22n2', 'o24')
        AND log_create_time >= '2025-05-01'
        AND X_Device_Country = 'KR'
        AND context_name = 'LSM'
        AND normal_log:app_id = 'com.disney.disneyplus-prod'
        AND normal_log:visible = true
),
-- Step 2: 첫 활동 월(코호트) 정의
user_first_activity AS (
    SELECT
        device_name,
        mac_addr,
        MIN(activity_date) AS cohort_date
    FROM user_dately_activity
    GROUP BY device_name, mac_addr
),
-- Step 3: 코호트 월로부터의 경과 월 계산
cohort_retention_data AS (
    SELECT
        fa.cohort_date,
        fa.device_name,
        fa.mac_addr,
        CAST(FLOOR(DATEDIFF(
            TO_DATE(ma.activity_date, 'yyyy-MM-dd'),
            TO_DATE(fa.cohort_date, 'yyyy-MM-dd')
        )) AS INT) AS date_number
    FROM user_first_activity fa
    JOIN user_dately_activity ma
      ON fa.mac_addr = ma.mac_addr AND fa.device_name = ma.device_name
    WHERE
        fa.cohort_date BETWEEN '2025-05-01' AND '2025-05-31'
),

-- Step 4: 코호트별, 월차별 잔존 유저 수(절대값)를 계산합니다. (중간 단계)
cohort_absolute_counts AS (
    SELECT
        cohort_date,
        device_name,
        COUNT(DISTINCT CASE WHEN date_number = 0 THEN mac_addr END) AS M0,
        COUNT(DISTINCT CASE WHEN date_number = 1 THEN mac_addr END) AS M1,
        COUNT(DISTINCT CASE WHEN date_number = 2 THEN mac_addr END) AS M2,
        COUNT(DISTINCT CASE WHEN date_number = 3 THEN mac_addr END) AS M3,
        COUNT(DISTINCT CASE WHEN date_number = 4 THEN mac_addr END) AS M4,
        COUNT(DISTINCT CASE WHEN date_number = 5 THEN mac_addr END) AS M5,
        COUNT(DISTINCT CASE WHEN date_number = 6 THEN mac_addr END) AS M6,
        COUNT(DISTINCT CASE WHEN date_number = 7 THEN mac_addr END) AS M7,
        COUNT(DISTINCT CASE WHEN date_number = 8 THEN mac_addr END) AS M8,
        COUNT(DISTINCT CASE WHEN date_number = 9 THEN mac_addr END) AS M9,
        COUNT(DISTINCT CASE WHEN date_number = 10 THEN mac_addr END) AS M10,
        COUNT(DISTINCT CASE WHEN date_number = 11 THEN mac_addr END) AS M11,
        COUNT(DISTINCT CASE WHEN date_number = 12 THEN mac_addr END) AS M12,
        COUNT(DISTINCT CASE WHEN date_number = 13 THEN mac_addr END) AS M13,
        COUNT(DISTINCT CASE WHEN date_number = 14 THEN mac_addr END) AS M14,
        COUNT(DISTINCT CASE WHEN date_number = 15 THEN mac_addr END) AS M15,
        COUNT(DISTINCT CASE WHEN date_number = 16 THEN mac_addr END) AS M16,
        COUNT(DISTINCT CASE WHEN date_number = 17 THEN mac_addr END) AS M17,
        COUNT(DISTINCT CASE WHEN date_number = 18 THEN mac_addr END) AS M18,
        COUNT(DISTINCT CASE WHEN date_number = 19 THEN mac_addr END) AS M19,
        COUNT(DISTINCT CASE WHEN date_number = 20 THEN mac_addr END) AS M20,
        COUNT(DISTINCT CASE WHEN date_number = 21 THEN mac_addr END) AS M21,
        COUNT(DISTINCT CASE WHEN date_number = 22 THEN mac_addr END) AS M22,
        COUNT(DISTINCT CASE WHEN date_number = 23 THEN mac_addr END) AS M23,
        COUNT(DISTINCT CASE WHEN date_number = 24 THEN mac_addr END) AS M24,
        COUNT(DISTINCT CASE WHEN date_number = 25 THEN mac_addr END) AS M25, 
        COUNT(DISTINCT CASE WHEN date_number = 26 THEN mac_addr END) AS M26,
        COUNT(DISTINCT CASE WHEN date_number = 27 THEN mac_addr END) AS M27,
        COUNT(DISTINCT CASE WHEN date_number = 28 THEN mac_addr END) AS M28,
        COUNT(DISTINCT CASE WHEN date_number = 29 THEN mac_addr END) AS M29,
        COUNT(DISTINCT CASE WHEN date_number = 30 THEN mac_addr END) AS M30
    FROM cohort_retention_data
    GROUP BY cohort_date, device_name
)

-- Step 5: 절대값 카운트를 바탕으로 잔존율(%)을 계산하여 최종 결과를 출력합니다.
SELECT
    cohort_date,
    device_name,
    100 as initial_rate,
    
        -- M+1 잔존율
    ROUND(M1 / M0 * 100, 2) AS M1_retention_rate,
    -- M+2 잔존율
    ROUND(M2 / M0 * 100, 2) AS M2_retention_rate,
    -- M+3 잔존율
    ROUND(M3 / M0 * 100, 2) AS M3_retention_rate,
    -- M+4 잔존율
    ROUND(M4 / M0 * 100, 2) AS M4_retention_rate,
    -- M+5 잔존율
    ROUND(M5 / M0 * 100, 2) AS M5_retention_rate,
    -- M+6 잔존율
    ROUND(M6 / M0 * 100, 2) AS M6_retention_rate,
    -- M+7 잔존율
    ROUND(M7 / M0 * 100, 2) AS M7_retention_rate,
    -- M+8 잔존율
    ROUND(M8 / M0 * 100, 2) AS M8_retention_rate,
    -- M+9 잔존율
    ROUND(M9 / M0 * 100, 2) AS M9_retention_rate,
    -- M+10 잔존율
    ROUND(M10 / M0 * 100, 2) AS M10_retention_rate,
    -- M+11 잔존율
    ROUND(M11 / M0 * 100, 2) AS M11_retention_rate,
    -- M+12 잔존율
    ROUND(M12 / M0 * 100, 2) AS M12_retention_rate,
    -- M+13 잔존율
    ROUND(M13 / M0 * 100, 2) AS M13_retention_rate,
    -- M+14 잔존율
    ROUND(M14 / M0 * 100, 2) AS M14_retention_rate,
    -- M+15 잔존율
    ROUND(M15 / M0 * 100, 2) AS M15_retention_rate,
    -- M+16 잔존율
    ROUND(M16 / M0 * 100, 2) AS M16_retention_rate,
    -- M+17 잔존율
    ROUND(M17 / M0 * 100, 2) AS M17_retention_rate,
    -- M+18 잔존율
    ROUND(M18 / M0 * 100, 2) AS M18_retention_rate,
    -- M+19 잔존율
    ROUND(M19 / M0 * 100, 2) AS M19_retention_rate,
    -- M+20 잔존율
    ROUND(M20 / M0 * 100, 2) AS M20_retention_rate,
    -- M+21 잔존율
    ROUND(M21 / M0 * 100, 2) AS M21_retention_rate,
    -- M+22 잔존율
    ROUND(M22 / M0 * 100, 2) AS M22_retention_rate,
    -- M+23 잔존율
    ROUND(M23 / M0 * 100, 2) AS M23_retention_rate,
    -- M+24 잔존율
    ROUND(M24 / M0 * 100, 2) AS M24_retention_rate,
    -- M+25 잔존율
    ROUND(M25 / M0 * 100, 2) AS M25_retention_rate,
    -- M+26 잔존율
    ROUND(M26 / M0 * 100, 2) AS M26_retention_rate,
    -- M+27 잔존율
    ROUND(M27 / M0 * 100, 2) AS M27_retention_rate,
    -- M+28 잔존율
    ROUND(M28 / M0 * 100, 2) AS M28_retention_rate,
    -- M+29 잔존율
    ROUND(M29 / M0 * 100, 2) AS M29_retention_rate,
    -- M+30 잔존율
    ROUND(M30 / M0 * 100, 2) AS M30_retention_rate,
    M0 as initial_users, 

    -- M+1 잔존 유저 수
    M1 AS M1_retained_users,
    -- M+2 잔존 유저 수
    M2 AS M2_retained_users,
    -- M+3 잔존 유저 수
    M3 AS M3_retained_users,
    -- M+4 잔존 유저 수
    M4 AS M4_retained_users,
    -- M+5 잔존 유저 수
    M5 AS M5_retained_users,
    -- M+6 잔존 유저 수
    M6 AS M6_retained_users,
    -- M+7 잔존 유저 수
    M7 AS M7_retained_users,
    -- M+8 잔존 유저 수
    M8 AS M8_retained_users,
    -- M+9 잔존 유저 수
    M9 AS M9_retained_users,
    -- M+10 잔존 유저 수
    M10 AS M10_retained_users,
    -- M+11 잔존 유저 수
    M11 AS M11_retained_users,
    -- M+12 잔존 유저 수
    M12 AS M12_retained_users,
    -- M+13 잔존 유저 수
    M13 AS M13_retained_users,
    -- M+14 잔존 유저 수
    M14 AS M14_retained_users,
    -- M+15 잔존 유저 수
    M15 AS M15_retained_users,
    -- M+16 잔존 유저 수
    M16 AS M16_retained_users,
    -- M+17 잔존 유저 수
    M17 AS M17_retained_users,
    -- M+18 잔존 유저 수
    M18 AS M18_retained_users,
    -- M+19 잔존 유저 수
    M19 AS M19_retained_users,
    -- M+20 잔존 유저 수
    M20 AS M20_retained_users,
    -- M+21 잔존 유저 수
    M21 AS M21_retained_users,
    -- M+22 잔존 유저 수
    M22 AS M22_retained_users,
    -- M+23 잔존 유저 수
    M23 AS M23_retained_users,
    -- M+24 잔존 유저 수
    M24 AS M24_retained_users,
    -- M+25 잔존 유저 수
    M25 AS M25_retained_users,
    -- M+26 잔존 유저 수
    M26 AS M26_retained_users,
    -- M+27 잔존 유저 수
    M27 AS M27_retained_users,
    -- M+28 잔존 유저 수
    M28 AS M28_retained_users,
    -- M+29 잔존 유저 수
    M29 AS M29_retained_users,
    -- M+30 잔존 유저 수
    M30 AS M30_retained_users
FROM cohort_absolute_counts
-- 0으로 나누는 오류 방지 (신규 유저가 없는 경우 해당 행 제외)
WHERE M0 > 0
ORDER BY cohort_date, device_name;